# Fine Tuning The Model

In [8]:
!pip install -q -U torch transformers accelerate bitsandbytes peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninsta

In [9]:
"""
LoRA fine-tuning of BioMistral-7B on lavita/medical-qa-datasets (all-processed)
Target: Kaggle 2x T4 (16GB each)

pip install transformers datasets accelerate peft bitsandbytes torch
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# -----------------------
# Config
# -----------------------
MODEL_NAME = "BioMistral/BioMistral-7B"
DATASET_NAME = "lavita/medical-qa-datasets"
DATASET_CONFIG = "all-processed"
OUTPUT_DIR = "./finetuned-model"
MAX_LENGTH = 1024

# -----------------------
# STEP 1 — Inspect the dataset BEFORE assuming a field name
# -----------------------
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(raw_dataset)
print(raw_dataset["train"].column_names)
print(raw_dataset["train"][0])

# >>> STOP HERE the first time you run this. Look at the printed column names
# and the sample row, then set TEXT_FIELD (or the instruction/output fields)
# below to match. The lines after this are written for a single text-style
# field — if it's actually instruction/input/output style, tell me the
# printed column names and I'll rewrite the tokenize_fn accordingly.

TEXT_FIELD = "text"  # <-- CHANGE THIS after checking the printout above

# -----------------------
# GPU check
# -----------------------
print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

# -----------------------
# 4-bit quantization config (QLoRA)
# -----------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# -----------------------
# Load tokenizer & model
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "13GiB", 1: "13GiB"},   # leaves headroom on each T4, avoids OOM
)
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

# -----------------------
# LoRA config
# -----------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# -----------------------
# Tokenize
# -----------------------
def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_FIELD],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

tokenized = raw_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)

if "validation" in tokenized:
    train_ds, eval_ds = tokenized["train"], tokenized["validation"]
else:
    split = tokenized["train"].train_test_split(test_size=0.05, seed=42)
    train_ds, eval_ds = split["train"], split["test"]

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# -----------------------
# Training arguments
# -----------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,      # start conservative on T4s
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
)

# -----------------------
# Train
# -----------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

trainer.train()

# -----------------------
# Save
# -----------------------
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"LoRA adapter saved to {OUTPUT_DIR}")

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', '__index_level_0__'],
        num_rows: 239357
    })
})
['instruction', 'input', 'output', '__index_level_0__']
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'hi. im a home health aide and i have a client with scoliosis in the back and kidney disease. her feet ankles and calves have been swollen for the past 2 weeks. mostly in her feet. she started a patch for pain in her legs 3 weeks ago. she started swelling up almost a week after she started the patch and the pain doctor cut the dose in half and she is still swollen. she has no blood clots in her legs because one of her doctors checked and they said it might be because of her back. what do you think? im concerned because she has been swollen up for to long and theres only so much i can do being her home health aide. we both want to get to the bottom of this swelling she i

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [1]:
import torch
print(torch.cuda.device_count())          # should print 2
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_name(1))

2
Tesla T4
Tesla T4
